In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
import os, sys, random
import numpy as np

# 再現性（任意）
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# パス
DATA_DIR = "dog_cat_photos"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR  = os.path.join(DATA_DIR, "test")

# 画像サイズ / バッチサイズ（MobileNetV2 は 224x224 を想定）
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# train をさらに train/val に分割（2割を検証用）
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode="binary",
    color_mode="rgb",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=0.2,
    subset="training",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode="binary",
    color_mode="rgb",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=0.2,
    subset="validation",
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    label_mode="binary",
    color_mode="rgb",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# データ拡張（学習時のみ適用）
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
    ],
    name="augmentation",
)

# パイプライン最適化
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

# MobileNetV2（ImageNet 事前学習）を転移学習で使用
base_model = MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False  # まずは凍結して学習

# モデル定義
inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)                # 画像水増し（学習時のみ動作）
x = preprocess_input(x)                      # MobileNetV2 用前処理 [-1,1] スケール
x = base_model(x, training=False)            # 事前学習特徴抽出
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)  # 2値分類

model = keras.Model(inputs, outputs, name="dog_vs_cat_mobilenetv2")

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [6]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=5, restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
)

Epoch 1/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 7s 433ms/step - accuracy: 0.6083 - loss: 0.7183 - val_accuracy: 0.7833 - val_loss: 0.5206
Epoch 2/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 276ms/step - accuracy: 0.8458 - loss: 0.3970 - val_accuracy: 0.9333 - val_loss: 0.3225
Epoch 3/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 305ms/step - accuracy: 0.9375 - loss: 0.2438 - val_accuracy: 0.9500 - val_loss: 0.2289
Epoch 4/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 300ms/step - accuracy: 0.9500 - loss: 0.1845 - val_accuracy: 0.9667 - val_loss: 0.1754
Epoch 5/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 364ms/step - accuracy: 0.9625 - loss: 0.1462 - val_accuracy: 0.9667 - val_loss: 0.1530
Epoch 6/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 551ms/step - accuracy: 0.9708 - loss: 0.1095 - val_accuracy: 0.9667 - val_loss: 0.1295
Epoch 7/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 502ms/step - accuracy: 0.9708 - loss: 0.1106 - val_accuracy: 0.9833 - val_loss: 0.1192
Epoch 8/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 453ms/step - accuracy: 0.9833 - loss: 0.0878 - val_accuracy: 0.9667 - val_loss:

In [7]:
print("=== テストデータで評価 ===")
test_loss, test_acc = model.evaluate(test_ds)
print(f"test_loss: {test_loss:.4f}  test_acc: {test_acc:.4f}")

=== テストデータで評価 ===
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 179ms/step - accuracy: 0.9800 - loss: 0.0772
test_loss: 0.0772  test_acc: 0.9800
